# Patient Adherence Risk Prediction

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Create a complete ML workflow** using only SQL in Snowflake
2. **Use `SNOWFLAKE.ML.CLASSIFICATION`** to build a binary classification model
3. **Generar datos sintéticos data** to simulate real-world patient behavior
4. **Engineer features** from multiple data sources
5. **Evaluate model performance** using built-in metrics
6. **Score new data** and generate predictions
7. **Build interactive dashboards** with Streamlit

---

## What You'll Build

A **binary classification model** that predicts whether patients will adhere to their medication regimen based on:
- Pharmacy refill patterns (fill gaps, days between fills)
- Patient demographics (age, insurance type)
- Engagement behaviors (app usage, portal logins)
- Treatment characteristics (medication complexity)

**Valor de Negocio**: Identify at-risk patients early for targeted intervention programs.

---

## Technical Requirements

- **Snowflake account** with standard SQL access
- **Cortex ML enabled** (contact your account team if needed)
- **Approximately 15 minutes** to complete
- **No Python knowledge required** - this is 100% SQL (except Streamlit dashboard)

---

## Key Snowflake Features Used

- `GENERATOR()` - Create realistic synthetic data
- `CREATE SNOWFLAKE.ML.CLASSIFICATION` - Entrenar clasificación binaria model
- `!PREDICT()` - Generar predictions on new data
- `!SHOW_EVALUATION_METRICS()` - View model performance
- Window functions - Calculate fill gaps and engagement trends
- Streamlit - Interactive dashboard (integrated in notebook)

Let's begin! 🚀

---

## Paso 1: Configuración del Entorno

### Qué Vamos a Hacer

Creating isolated Snowflake objects para este tutorial:
- **Database**: `PATIENT_ADHERENCE_DB` - Container for all tables and models
- **Schema**: `PUBLIC` - Namespace within the database
- **Warehouse**: `COMPUTE_WH` - Compute resources for queries

### Why This Matters

Using dedicated objects keeps this tutorial isolated from your production environment and makes cleanup easy if needed.

### Snowflake Concepts

- **Database**: Logical container for schemas (like a folder)
- **Schema**: Namespace for tables, views, and ML models
- **Warehouse**: Virtual compute cluster that executes queries (can auto-suspend when idle)
- **Context**: The active database/schema/warehouse for your session

In [ ]:
-- Crear una base de datos dedicada para este tutorial
CREATE DATABASE IF NOT EXISTS PATIENT_ADHERENCE_DB;

-- Create a schema within the database
CREATE SCHEMA IF NOT EXISTS PATIENT_ADHERENCE_DB.PUBLIC;

-- Set our session context to use this database and schema
USE SCHEMA PATIENT_ADHERENCE_DB.PUBLIC;

-- Create a small compute warehouse for running queries
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'  -- Smallest size (cost-effective)
    AUTO_SUSPEND = 60              -- Turn off after 60 seconds of inactivity
    AUTO_RESUME = TRUE;            -- Automatically turn on when needed

-- Set this warehouse as active for our session
USE WAREHOUSE COMPUTE_WH;

-- Verify our environment is configured correctly
SELECT 
    CURRENT_DATABASE() as database_name,
    CURRENT_SCHEMA() as schema_name,
    CURRENT_WAREHOUSE() as warehouse_name;

---

## Paso 2: Generar Datos Sintéticos Patient Data

### Qué Vamos a Hacer

Creating realistic synthetic data for **500 patients** across three data sources:
1. **Patient Demographics** - Age, insurance, location
2. **Pharmacy Fills** - Prescription refill history
3. **App Usage** - Mobile app engagement patterns

### The `GENERATOR()` Function

`GENERATOR(ROWCOUNT => N)` is Snowflake's powerful function for creating synthetic data:
- Generates N rows instantly
- Combined with `UNIFORM()`, `SEQ4()`, and date functions to create realistic patterns
- No external data files needed!

### Data Patterns

We're simulating realistic behavior:
- **Adherent patients** (60%): Regular refills, high app usage
- **Non-adherent patients** (40%): Fill gaps, declining engagement
- Age distribution: 25-75 years
- Insurance types: Commercial, Medicare, Medicaid

This synthetic data is designed for demonstration - replace with your real data in production.

In [ ]:
-- Generar 500 synthetic patients
CREATE OR REPLACE TABLE patients AS
SELECT 
    'P' || LPAD(SEQ4(), 5, '0') as patient_id,
    UNIFORM(25, 75, RANDOM()) as age,
    CASE WHEN UNIFORM(0, 1, RANDOM()) < 0.5 THEN 'M' ELSE 'F' END as gender,
    CASE 
        WHEN UNIFORM(1, 100, RANDOM()) <= 50 THEN 'Commercial'
        WHEN UNIFORM(1, 100, RANDOM()) <= 80 THEN 'Medicare'
        ELSE 'Medicaid'
    END as insurance_type,
    DATEADD(day, -UNIFORM(180, 730, RANDOM()), CURRENT_DATE()) as enrollment_date,
    -- Ground truth: 60% adherent, 40% non-adherent
    CASE WHEN UNIFORM(0, 1, RANDOM()) < 0.6 THEN 1 ELSE 0 END as is_adherent
FROM TABLE(GENERATOR(ROWCOUNT => 500));

-- Preview the generated patients
SELECT * FROM patients LIMIT 10;

In [ ]:
-- Generar pharmacy fill records (multiple fills per patient)
CREATE OR REPLACE TABLE pharmacy_fills AS
WITH fill_sequence AS (
    SELECT 
        SEQ4() - 1 as fill_seq  -- 0-indexed sequence
    FROM TABLE(GENERATOR(ROWCOUNT => 8))
)
SELECT 
    p.patient_id,
    DATEADD(day, fs.fill_seq * 
        CASE 
            -- Adherent patients: Regular 30-day refills
            WHEN p.is_adherent = 1 THEN UNIFORM(28, 32, RANDOM())
            -- Non-adherent: Longer gaps (35-60 days)
            ELSE UNIFORM(35, 60, RANDOM())
        END,
        p.enrollment_date
    ) as fill_date,
    'Xarelto 20mg' as medication,
    30 as days_supply,
    fs.fill_seq + 1 as refill_number,
    UNIFORM(10.00, 50.00, RANDOM())::DECIMAL(10,2) as copay_amount
FROM patients p
CROSS JOIN fill_sequence fs
WHERE DATEADD(day, fs.fill_seq * 35, p.enrollment_date) <= CURRENT_DATE();

-- Preview pharmacy fills
SELECT * FROM pharmacy_fills 
WHERE patient_id = 'P00001'
ORDER BY fill_date;

In [ ]:
-- Generar app usage data (daily engagement patterns)
CREATE OR REPLACE TABLE app_usage AS
WITH day_sequence AS (
    SELECT 
        SEQ4() - 1 as day_seq  -- 0-indexed sequence
    FROM TABLE(GENERATOR(ROWCOUNT => 120))
)
SELECT 
    p.patient_id,
    DATEADD(day, ds.day_seq, p.enrollment_date) as usage_date,
    CASE 
        -- Adherent patients: Altoer engagement
        WHEN p.is_adherent = 1 THEN UNIFORM(1, 5, RANDOM())
        -- Non-adherent: Bajoer, declining engagement
        ELSE GREATEST(0, UNIFORM(0, 3, RANDOM()) - (ds.day_seq / 30))
    END as session_count,
    CASE 
        WHEN p.is_adherent = 1 THEN UNIFORM(5, 20, RANDOM())
        ELSE UNIFORM(0, 8, RANDOM())
    END as minutes_active
FROM patients p
CROSS JOIN day_sequence ds
WHERE DATEADD(day, ds.day_seq, p.enrollment_date) <= CURRENT_DATE()
  AND UNIFORM(0, 1, RANDOM()) < 0.7;  -- Not every patient uses app every day

-- Preview app usage
SELECT * FROM app_usage 
WHERE patient_id = 'P00001'
ORDER BY usage_date DESC
LIMIT 10;

---

## Paso 3: Ingeniería de Variables

### Qué Vamos a Hacer

Transformando datos brutos into **ML-ready features** that predict adherence:

1. **Pharmacy Fill Patterns**:
   - `avg_days_between_fills` - How regularly they refill (adherent = ~30 days)
   - `fill_gap_days` - Days since last fill (adherent = small gaps)
   - `total_fills` - Number of refills (adherent = more fills)

2. **App Engagement**:
   - `avg_sessions_per_day` - Daily app usage (adherent = higher)
   - `total_active_minutes` - Cumulative engagement time
   - `days_since_last_login` - Recency of engagement

3. **Patient Demographics**:
   - `age` - Older patients may have different adherence patterns
   - `insurance_type` - Coverage affects adherence
   - `enrollment_days` - How long they've been enrolled

### Window Functions

We use `LAG()` to calculate **days between fills**:
```sql
LAG(fill_date) OVER (PARTITION BY patient_id ORDER BY fill_date)
```

This looks at the previous fill date for each patient to calculate gaps.

### Why Ingeniería de Variables Matters

Raw data (fill dates, login times) isn't useful for ML. Features (gaps, trends, patterns) are what models learn from!

In [ ]:
-- Calculate fill gaps using window functions (separate CTE)
CREATE OR REPLACE TABLE fill_gaps AS
SELECT 
    patient_id,
    fill_date,
    DATEDIFF('day', 
        LAG(fill_date) OVER (PARTITION BY patient_id ORDER BY fill_date),
        fill_date
    ) as days_since_last_fill
FROM pharmacy_fills;

-- Create feature table with all engineered features
CREATE OR REPLACE TABLE patient_features AS
SELECT 
    p.patient_id,
    
    -- Demographics
    p.age::FLOAT as age,
    CASE WHEN p.insurance_type = 'Commercial' THEN 1 ELSE 0 END::FLOAT as is_commercial,
    CASE WHEN p.insurance_type = 'Medicare' THEN 1 ELSE 0 END::FLOAT as is_medicare,
    DATEDIFF('day', p.enrollment_date, CURRENT_DATE())::FLOAT as enrollment_days,
    
    -- Pharmacy fill features (from fill_gaps CTE)
    COUNT(DISTINCT pf.fill_date)::FLOAT as total_fills,
    AVG(fg.days_since_last_fill)::FLOAT as avg_days_between_fills,
    MAX(DATEDIFF('day', pf.fill_date, CURRENT_DATE()))::FLOAT as days_since_last_fill,
    AVG(pf.copay_amount)::FLOAT as avg_copay,
    
    -- App usage features
    COALESCE(AVG(au.session_count), 0)::FLOAT as avg_sessions_per_day,
    COALESCE(SUM(au.minutes_active), 0)::FLOAT as total_active_minutes,
    COALESCE(MAX(DATEDIFF('day', au.usage_date, CURRENT_DATE())), 999)::FLOAT as days_since_last_login,
    
    -- Target variable
    p.is_adherent
    
FROM patients p
LEFT JOIN pharmacy_fills pf ON p.patient_id = pf.patient_id
LEFT JOIN fill_gaps fg ON pf.patient_id = fg.patient_id AND pf.fill_date = fg.fill_date
LEFT JOIN app_usage au ON p.patient_id = au.patient_id
GROUP BY p.patient_id, p.age, p.insurance_type, p.enrollment_date, p.is_adherent;

-- Preview engineered features
SELECT * FROM patient_features LIMIT 10;

---

## Paso 4: Preparar Datos de Entrenamiento

### Qué Vamos a Hacer

Dividiendo nuestros 500 patients into:
- **Training set** (80% = 400 patients) - Used to train the model
- **Test set** (20% = 100 patients) - Used to evaluate performance

### Why Split Data?

We need to test the model on **unseen data** to ensure it generalizes well. Training and testing on the same data would give artificially high accuracy.

### The `SAMPLE` Function

`SAMPLE (80)` randomly selects 80% of rows for training. The remaining 20% becomes our test set.

### What Goes into Training?

- **Features** (predictors): All columns except `patient_id` and `is_adherent`
- **Target** (label): `is_adherent` (0 or 1)

In [ ]:
-- Crear conjunto de entrenamiento (80% of patients)
CREATE OR REPLACE TABLE training_data AS
SELECT * FROM patient_features
SAMPLE (80);

-- Crear conjunto de test (remaining 20%)
CREATE OR REPLACE TABLE test_data AS
SELECT * FROM patient_features
WHERE patient_id NOT IN (SELECT patient_id FROM training_data);

-- Verify split sizes
SELECT 
    'Training' as dataset,
    COUNT(*) as patient_count,
    SUM(is_adherent) as adherent_count,
    ROUND(SUM(is_adherent) / COUNT(*) * 100, 1) || '%' as adherent_rate
FROM training_data

UNION ALL

SELECT 
    'Test' as dataset,
    COUNT(*) as patient_count,
    SUM(is_adherent) as adherent_count,
    ROUND(SUM(is_adherent) / COUNT(*) * 100, 1) || '%' as adherent_rate
FROM test_data;

---

## Paso 5: Entrenar el Modelo de Clasificación

### Qué Vamos a Crear

A **binary classification model** that predicts whether a patient will be adherent (1) or non-adherent (0) based on their behavioral features.

### Understanding `SNOWFLAKE.ML.CLASSIFICATION`

This is Snowflake's **native AutoML function** that trains machine learning models using only SQL:

```sql
CREATE SNOWFLAKE.ML.CLASSIFICATION model_name(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'training_data'),
    TARGET_COLNAME => 'is_adherent',
    CONFIG_OBJECT => {'evaluate': TRUE}
);
```

### Why ML.CLASSIFICATION for Adherence Prediction?

`SNOWFLAKE.ML.CLASSIFICATION` is perfect for this use case because:

- **AutoML Magic**: Automatically tests multiple algorithms (XGBoost, Random Forest, Logistic Regression)
- **No Python Required**: 100% SQL - accessible to analysts and business users
- **Automatic Feature Selection**: Identifies which features are most predictive
- **Built-in Evaluation**: Tests performance on holdout data automatically
- **Scalable**: Handles millions of patients without code changes

### How It Works (Behind the Scenes)

When you call `CREATE SNOWFLAKE.ML.CLASSIFICATION`, Snowflake:

1. **Analyzes Your Data**: Examines feature distributions, missing values, correlations
2. **Trains Multiple Algorithms**: Tests 3-5 different ML algorithms in parallel
3. **Hyperparameter Tuning**: Optimizes each algorithm's settings automatically
4. **Selects Best Model**: Chooses the algorithm with highest accuracy
5. **Creates Prediction Function**: Returns a model with `!PREDICT()` method

**All of this happens automatically** - you just provide data and target column!

### Real-World Example

**Input Features** (for Patient #12345):
- `days_since_enrollment`: 180
- `total_fills`: 5
- `avg_days_between_fills`: 32
- `days_since_last_fill`: 45
- `avg_copay`: 25.00
- `avg_sessions_per_day`: 0.8
- `total_active_minutes`: 240
- `days_since_last_login`: 7

**Training Process**:
- Model learns: "Patients with `days_since_last_fill` > 40 AND `avg_sessions_per_day` < 1.0 are 73% likely to be non-adherent"
- Discovers patterns: Alto copays + long fill gaps = high risk

**Result**: Model can now predict adherence for new patients!

### Key Parameters Explained

**`INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'training_data')`**
- Passes reference to your training table (not a copy)
- `SYSTEM$REFERENCE()` allows model to inherit your privileges
- Table must include both features and target column

**`TARGET_COLNAME => 'is_adherent'`**
- The column you want to predict (0 or 1 for binary classification)
- Must be INTEGER or BOOLEAN type
- Model learns patterns that lead to 0 vs 1

**`CONFIG_OBJECT => {'evaluate': TRUE}`**
- **`evaluate`**: Automatically test model performance (recommended!)
- Optional configs: `on_error`, `imbalance_method`, `classification_type`

### What Gets Created?

After training, you get a **model object** (`adherence_predictor`) with methods:

- **`!PREDICT()`**: Generar predictions for new patients
- **`!SHOW_EVALUATION_METRICS()`**: View accuracy, precision, recall
- **`!SHOW_CONFUSION_MATRIX()`**: See prediction errors
- **`!SHOW_FEATURE_IMPORTANCE()`**: Identify most important features
- **`!SHOW_TRAINING_LOGS()`**: Debug training process

### Training Time

- **400 patients**: 30-60 seconds
- **10,000 patients**: 2-4 minutes
- **1 million patients**: 10-20 minutes

Training is a one-time cost - predictions are instant!

### Technical Details

**Supported Algorithms** (Snowflake tests all and picks best):
- **XGBoost**: Gradient boosting (often wins on structured data)
- **Random Forest**: Ensemble of decision trees
- **Logistic Regression**: Linear classifier (fast, interpretable)
- **Neural Networks**: Deep learning (for complex patterns)

**Feature Requirements**:
- All features must be numeric (FLOAT, INTEGER, DECIMAL)
- NULL values are handled automatically
- No need to normalize or scale features

**Model Storage**:
- Stored as Snowflake object (like a table or view)
- Includes trained weights, preprocessing, and metadata
- Secured with role-based access control

In [ ]:
-- Entrenar clasificación binaria model
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION adherence_predictor(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'training_data'),
    TARGET_COLNAME => 'is_adherent',  -- Correct parameter name
    CONFIG_OBJECT => {'evaluate': TRUE}
);

-- Training complete! The model is now ready to use.

---

## Paso 6: Evaluar Rendimiento del Modelo

### Qué Vamos a Verificar

Qué tan bien funciona nuestro modelo predict adherence?

### Key Metrics

- **Accuracy**: % of correct predictions overall (target: >70%)
- **Precision**: Of patients predicted as adherent, % who actually are (target: >75%)
- **Recall**: Of actual adherent patients, % we correctly identified (target: >80%)
- **F1 Score**: Harmonic mean of precision and recall (target: >0.75)

### Confusion Matrix

Shows where the model makes mistakes:
- **True Positives (TP)**: Correctly predicted adherent
- **True Negatives (TN)**: Correctly predicted non-adherent
- **False Positives (FP)**: Predicted adherent, but actually non-adherent
- **False Negatives (FN)**: Predicted non-adherent, but actually adherent

### What's "Good Enough"?

For patient adherence prediction:
- Alto **recall** is critical (don't miss at-risk patients)
- Moderate **precision** is acceptable (some false alarms are OK)
- Overall **accuracy** >70% is good for this use case

In [ ]:
-- Ver métricas de evaluación (accuracy, precision, recall, F1-score, AUC)
CALL adherence_predictor!SHOW_EVALUATION_METRICS();

In [ ]:
-- Ver métricas for each class (0 = non-adherent, 1 = adherent)
CALL adherence_predictor!SHOW_EVALUATION_METRICS();

In [ ]:
-- Ver matriz de confusión
CALL adherence_predictor!SHOW_CONFUSION_MATRIX();

---

## Paso 7: Puntuar Nuevos Patients

### Qué Vamos a Hacer

Usando nuestro modelo entrenado to **predict adherence** for the test set (patients the model hasn't seen).

### The `!PREDICT()` Method

Snowflake ML models expose a `!PREDICT()` method that:
1. Takes input features from a table
2. Returns predictions in a VARIANT object with:
   - **`class`**: Predicted class (0 or 1)
   - **`probability`**: Confidence scores for each class
     - `probability['0']` - Probability of non-adherent
     - `probability['1']` - Probability of adherent

### Understanding Predictions

The model returns a **probability** for each class. For example:
```json
{"class": 1, "probability": {"0": 0.15, "1": 0.85}}
```
This means: **85% confident** the patient is adherent (class 1).

### Risk Stratification

We'll categorize patients into risk tiers:
- **Alto Riesgo**: <30% adherence probability → Immediate intervention
- **Riesgo Medio**: 30-70% probability → Monitor closely
- **Bajo Riesgo**: >70% probability → Standard care

### Extracting from VARIANT

Use bracket notation to access nested JSON:
```sql
prediction['probability']['1']::FLOAT  -- Get probability for class 1
```

In [ ]:
-- Puntuar test patients and extract predictions
CREATE OR REPLACE TABLE adherence_predictions AS
SELECT 
    patient_id,
    
    -- Extract predicted class and probabilities from VARIANT object
    adherence_predictor!PREDICT(
        OBJECT_CONSTRUCT(
            'age', age,
            'is_commercial', is_commercial,
            'is_medicare', is_medicare,
            'enrollment_days', enrollment_days,
            'total_fills', total_fills,
            'avg_days_between_fills', avg_days_between_fills,
            'days_since_last_fill', days_since_last_fill,
            'avg_copay', avg_copay,
            'avg_sessions_per_day', avg_sessions_per_day,
            'total_active_minutes', total_active_minutes,
            'days_since_last_login', days_since_last_login
        )
    ) as prediction,
    
    -- Extract predicted class (0 or 1)
    prediction['class']::INT as predicted_adherent,
    
    -- Extract probability of being adherent (class 1)
    prediction['probability']['1']::FLOAT as adherence_probability,
    
    -- Risk stratification based on probability
    CASE 
        WHEN prediction['probability']['1']::FLOAT < 0.30 THEN 'High Risk'
        WHEN prediction['probability']['1']::FLOAT < 0.70 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END as risk_tier,
    
    -- Actual adherence (for validation)
    is_adherent as actual_adherent
    
FROM test_data;

-- Preview predictions
SELECT 
    patient_id,
    predicted_adherent,
    ROUND(adherence_probability, 3) as probability,
    risk_tier,
    actual_adherent
FROM adherence_predictions
ORDER BY adherence_probability
LIMIT 20;

In [ ]:
-- Analizar predicciones by risk tier
SELECT 
    risk_tier,
    COUNT(*) as patient_count,
    ROUND(AVG(adherence_probability), 3) as avg_probability,
    SUM(CASE WHEN predicted_adherent = actual_adherent THEN 1 ELSE 0 END) as correct_predictions,
    ROUND(SUM(CASE WHEN predicted_adherent = actual_adherent THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) || '%' as accuracy
FROM adherence_predictions
GROUP BY risk_tier
ORDER BY risk_tier DESC;

---

## Paso 8: Dashboard Interactivo

### Lo Que Verás

An interactive **Streamlit dashboard** showing:
- Model performance metrics
- Risk distribution across patients
- Patient-level predictions with filters

### How to Use

The dashboard below is **embedded in this notebook** - no separate deployment needed!

Run this cell to launch the dashboard.

In [ ]:
# Dashboard Streamlit for Patient Adherence Predictions
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

# Get Snowpark session
session = get_active_session()

st.title("🏥 Patient Adherence Risk Prediction Dashboard")
st.markdown("### ML-Powered Adherence Risk Scoring")

# Load predictions
predictions_df = session.table("adherence_predictions").to_pandas()

# Key Metrics
col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric("Total Patients", len(predictions_df))

with col2:
    high_risk = len(predictions_df[predictions_df['RISK_TIER'] == 'High Risk'])
    st.metric("High Risk", high_risk)

with col3:
    accuracy = (predictions_df['PREDICTED_ADHERENT'] == predictions_df['ACTUAL_ADHERENT']).mean()
    st.metric("Model Accuracy", f"{accuracy:.1%}")

with col4:
    avg_prob = predictions_df['ADHERENCE_PROBABILITY'].mean()
    st.metric("Avg Adherence Prob", f"{avg_prob:.2%}")

st.markdown("---")

# Risk Distribution Bar Chart
st.subheader("📊 Risk Distribution")
risk_counts = predictions_df['RISK_TIER'].value_counts().sort_index(ascending=False)
# Convert to DataFrame with proper column names to avoid Altair warnings
risk_df = pd.DataFrame({
    'Count': risk_counts.values
}, index=risk_counts.index)
st.bar_chart(risk_df)

# Probability Distribution
st.subheader("📈 Adherence Probability Distribution")
# Use histogram with explicit bins
prob_bins = pd.cut(predictions_df['ADHERENCE_PROBABILITY'], bins=20)
prob_counts = prob_bins.value_counts().sort_index()
# Create clean DataFrame for charting
prob_df = pd.DataFrame({
    'Frequency': prob_counts.values
}, index=range(len(prob_counts)))
st.bar_chart(prob_df)

st.markdown("---")

# Patient-Level Predictions
st.subheader("👤 Patient-Level Predictions")

# Filter by risk tier
risk_filter = st.multiselect(
    "Filter by Risk Tier",
    options=['High Risk', 'Medium Risk', 'Low Risk'],
    default=['High Risk', 'Medium Risk', 'Low Risk']
)

filtered_df = predictions_df[predictions_df['RISK_TIER'].isin(risk_filter)].copy()

# Create display dataframe with formatted values
display_df = filtered_df[[
    'PATIENT_ID',
    'ADHERENCE_PROBABILITY',
    'RISK_TIER',
    'PREDICTED_ADHERENT',
    'ACTUAL_ADHERENT'
]].copy()  # Create a proper copy

# Sort before formatting
display_df = display_df.sort_values('ADHERENCE_PROBABILITY')

# Format probability as percentage in a new column to avoid dtype issues
display_df['Adherence %'] = display_df['ADHERENCE_PROBABILITY'].apply(lambda x: f"{x:.1%}")

# Display with formatted column, drop the original float column
st.dataframe(
    display_df[['PATIENT_ID', 'Adherence %', 'RISK_TIER', 'PREDICTED_ADHERENT', 'ACTUAL_ADHERENT']],
    use_container_width=True,
    height=400
)

# Summary by risk tier
st.markdown("---")
st.subheader("📋 Summary by Risk Tier")

summary_data = []
for tier in ['High Risk', 'Medium Risk', 'Low Risk']:
    tier_df = predictions_df[predictions_df['RISK_TIER'] == tier]
    if len(tier_df) > 0:
        count = len(tier_df)
        avg_prob = tier_df['ADHERENCE_PROBABILITY'].mean()
        correct = (tier_df['PREDICTED_ADHERENT'] == tier_df['ACTUAL_ADHERENT']).sum()
        accuracy = (correct / count * 100) if count > 0 else 0
        
        summary_data.append({
            'Risk Tier': tier,
            'Patient Count': count,
            'Avg Probability': f"{avg_prob:.1%}",
            'Correct Predictions': correct,
            'Accuracy': f"{accuracy:.1f}%"
        })

summary = pd.DataFrame(summary_data)
st.dataframe(summary, use_container_width=True)

# Download predictions
st.markdown("---")
st.subheader("💾 Export Results")
csv = filtered_df.to_csv(index=False)
st.download_button(
    label="Download Predictions as CSV",
    data=csv,
    file_name="adherence_predictions.csv",
    mime="text/csv"
)

st.markdown("---")
st.caption("Built with Snowflake Cortex ML + Streamlit | Data: Synthetic")

---

## Paso 9: Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `PATIENT_ADHERENCE_DB` database (and all tables/models within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

⚠️ **Warning**: This action cannot be undone. All data and models will be permanently deleted.


In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS PATIENT_ADHERENCE_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;